In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import f1_score

In [2]:
device = torch.device("mps")

In [3]:
class WaferCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 14 * 14, 9)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.fc(x)
        return x
        
model = WaferCNN()

In [6]:
X = np.load("/Users/a./Wafer_project/wafer-defects/data/X.npy")
y = np.load("/Users/a./Wafer_project/wafer-defects/data/y.npy")


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=67
)

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=68
)
print(X_train.shape, X_val.shape)

(24152, 64, 64) (4263, 64, 64)


In [9]:
classes = np.unique(y).tolist()
class_to_idx = {name: i for i, name in enumerate(classes)}

In [10]:
y_train_idx = np.array([class_to_idx[c] for c in y_train])
y_val_idx = np.array([class_to_idx[c] for c in y_val])
y_test_idx = np.array([class_to_idx[c] for c in y_test])

In [11]:
X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val_t = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

y_train_t = torch.tensor(y_train_idx, dtype=torch.long)
y_val_t = torch.tensor(y_val_idx, dtype=torch.long)
y_test_t = torch.tensor(y_test_idx, dtype=torch.long)

In [12]:
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [14]:
counts = torch.bincount(y_train_t)
weights = len(y_train_t) / (9 * counts)

In [16]:
print(len(train_loader), len(val_loader), len(test_loader))

378 67 111


In [19]:
def predict(model, loader):
    all_preds = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            all_preds.append(logits.argmax(dim=1).cpu())
    return torch.cat(all_preds).numpy()
    

In [22]:
def train_model(p, epochs = 5):
    torch.manual_seed(111)
    model = WaferCNN().to(device)
    loss_fn = nn.CrossEntropyLoss(weight = (weights ** p).to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        running_total = 0
        for xb,yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_total += loss.item()
        print(f"epoch{epoch+1}:{running_total/len(train_loader):.3f}")
    return model


In [26]:
powers = [0, 0.25, 0.5, 0.75, 1]
for p in powers:
    model = train_model(p, epochs = 5)
    val_preds = predict(model, val_loader)
    print(p,f1_score(y_val_t, val_preds, average="macro"))

epoch1:0.857
epoch2:0.587
epoch3:0.500
epoch4:0.441
epoch5:0.404
0 0.7497995067917336
epoch1:0.997
epoch2:0.677
epoch3:0.583
epoch4:0.513
epoch5:0.455
0.25 0.7457338705021503
epoch1:1.031
epoch2:0.695
epoch3:0.597
epoch4:0.524
epoch5:0.455
0.5 0.7170778294774718
epoch1:1.133
epoch2:0.732
epoch3:0.643
epoch4:0.564
epoch5:0.505
0.75 0.7413938144921497
epoch1:1.149
epoch2:0.718
epoch3:0.617
epoch4:0.545
epoch5:0.467
1 0.7328416120608694
